# Phase A/B on Kaggle

This notebook is a **thin wrapper** around the `aiforensics` command-line
interface that already lives in this repository. It installs the package,
prepares an environment, points the pipeline at storage you control, and then
calls the same public CLI you would run locally.

It does **not** implement dataset parsing, manifest validation, model loading,
inference, metrics, reporting, caching, or NPR checkout. Those behaviours belong
to the package (Tasks 1-11) and must stay there.

Before you start, understand the limits:

- A full Phase A/B comparison needs **pre-provisioned images and manifests**.
  `aiforensics prepare` validates what already exists; it is not a
  research-dataset downloader.
- The heavy baselines (Qwen-VL, Assisted Qwen, NPR) generally need **CUDA**,
  network access for model weights, and an operator-provided NPR checkpoint.
- The optional smoke section proves the pipeline works in this environment.
  Smoke metrics are pipeline checks and **not scientific evidence**.
- Attached datasets under `/kaggle/input` are **read-only**. Every generated
  artifact must be written to writable storage such as `/kaggle/working`, which
  is also **ephemeral** once the session ends unless you save the output.
- Installing packages and provisioning Python 3.10 needs the notebook
  **Internet** setting to be enabled.

See `docs/runbook-colab-kaggle.md` for the operator runbook.

## 1. Runtime preflight

This cell only reports what the environment looks like. It deliberately does
**not** fail when the notebook kernel is newer than Python 3.10: the kernel never
imports `aiforensics`, the CLI does. What matters is whether a Python 3.10
interpreter is available for the CLI, because `pyproject.toml` declares

```text
requires-python = ">=3.10,<3.11"
```

GPU output below is informational. Real device selection and deferral stay
inside the baseline adapters.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# User-editable: where the Python 3.10 environment for the CLI lives.
CLI_VENV_PATH = Path("/kaggle/working/aiforensics-venv310")

TARGET_PY = (3, 10)


def _venv_bin(venv_path: Path) -> Path:
    """Return the scripts directory of a virtual environment."""
    return venv_path / ("Scripts" if os.name == "nt" else "bin")


def _interpreter_version(executable: str) -> tuple[int, int] | None:
    """Return (major, minor) for an interpreter, or None when unusable."""
    try:
        result = subprocess.run(
            [executable, "-c", "import sys; print(sys.version_info[0], sys.version_info[1])"],
            capture_output=True,
            text=True,
            check=False,
        )
    except OSError:
        return None
    if result.returncode != 0:
        return None
    parts = result.stdout.split()
    if len(parts) != 2:
        return None
    return int(parts[0]), int(parts[1])


def find_cli_python() -> str | None:
    """Find an interpreter that satisfies the repository Python contract."""
    candidates = [
        str(_venv_bin(CLI_VENV_PATH) / "python"),
        shutil.which("python3.10"),
        sys.executable,
    ]
    for candidate in candidates:
        if not candidate or not Path(candidate).exists():
            continue
        if _interpreter_version(candidate) == TARGET_PY:
            return candidate
    return None


print("notebook kernel:", sys.version.split()[0], "(informational only)")
print("working directory:", Path.cwd())

CLI_PYTHON = find_cli_python()
if CLI_PYTHON:
    print("Python 3.10 for the CLI:", CLI_PYTHON)
else:
    print(
        "No Python 3.10 interpreter found yet.\n"
        "Run the optional provisioning cell in section 2 before installing the package."
    )

gpu = shutil.which("nvidia-smi")
if gpu:
    subprocess.run([gpu], check=False)
else:
    print("nvidia-smi not found: no GPU visible to this runtime (informational).")

## 2. Optional: provision Python 3.10 for the CLI

Run this section **only when section 1 reported no Python 3.10 interpreter**.

It creates a dedicated virtual environment on Python 3.10 and puts it first on
`PATH`, so later cells can call `aiforensics` unchanged. This satisfies the
repository's `requires-python` contract honestly: the package is installed under
a real 3.10 interpreter. Never edit `pyproject.toml` to make an install succeed.

This step needs **network access** (to fetch `uv`, the interpreter, and the
dependencies).

In [ ]:
def provision_cli_python(venv_path: Path) -> str:
    """Create a Python 3.10 virtual environment and prepend it to PATH."""
    if shutil.which("uv") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "uv"],
            check=True,
        )

    uv = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
    subprocess.run([uv, "python", "install", "3.10"], check=True)
    subprocess.run(
        [uv, "venv", "--seed", "--no-project", "--python", "3.10", str(venv_path)],
        check=True,
    )

    bin_dir = _venv_bin(venv_path)
    os.environ["PATH"] = f"{bin_dir}{os.pathsep}{os.environ.get('PATH', '')}"
    os.environ["VIRTUAL_ENV"] = str(venv_path)
    return str(bin_dir / "python")


CLI_PYTHON = provision_cli_python(CLI_VENV_PATH)
print("provisioned CLI interpreter:", CLI_PYTHON)

### 2b. Verify the CLI interpreter

This is the first hard gate. If no valid Python 3.10 environment exists after
provisioning, stop here and fix the environment instead of working around the
version contract.

In [ ]:
resolved = shutil.which("python") or ""
version = _interpreter_version(resolved) if resolved else None

print("python on PATH:", resolved or "<none>")
print("python version:", ".".join(str(p) for p in version) if version else "<unknown>")

if version != TARGET_PY:
    raise RuntimeError(
        "No usable Python 3.10 environment for the CLI. The repository requires "
        ">=3.10,<3.11. Run the provisioning cell above, or select a runtime that "
        "provides Python 3.10. Do not modify pyproject.toml to bypass this."
    )

print("OK: the CLI will run under Python 3.10.")

## 3. Repository location

Point `REPO_ROOT` at a checkout of this repository. Either attach it as a Kaggle
dataset/utility script and copy it into writable storage, or set `REPO_GIT_URL`
to a repository you control and let the cell clone it (needs Internet enabled).
Do not embed credentials in this notebook; use an environment variable or Kaggle
Secrets if a private clone needs authentication.

In [ ]:
# User-editable inputs.
REPO_ROOT = Path("/kaggle/working/ai-image-forensics")
REPO_GIT_URL = ""  # e.g. "https://github.com/<owner>/<repo>.git"; leave empty to skip cloning

if not REPO_ROOT.exists() and REPO_GIT_URL:
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_ROOT)], check=True)

REQUIRED_REPO_FILES = [
    REPO_ROOT / "pyproject.toml",
    REPO_ROOT / "configs/phase_ab.yaml",
]
missing = [str(path) for path in REQUIRED_REPO_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "REPO_ROOT does not look like this repository. Missing: "
        + ", ".join(missing)
        + ". Set REPO_ROOT (or REPO_GIT_URL) to a valid checkout."
    )

os.chdir(REPO_ROOT)
print("repository root:", REPO_ROOT)

## 4. Install the package

Dependency names come from `pyproject.toml`; nothing is pinned again here. The
optional extras map to the baselines: `clip` for the CLIP probe, `qwen` for
Qwen-VL and Assisted Qwen, `npr` for the NPR runtime bridge.

Model weights and the NPR checkpoint are **not** packaged with the repository.

In [ ]:
os.environ["AIF_REPO_ROOT"] = str(REPO_ROOT)
print("AIF_REPO_ROOT =", os.environ["AIF_REPO_ROOT"])

In [ ]:
%%bash
set -euo pipefail

cd "$AIF_REPO_ROOT"
python -m pip install --quiet --upgrade pip
python -m pip install -e ".[clip,qwen,npr]"

### 4b. Verify the CLI resolves from the Python 3.10 environment

Second hard gate: the `aiforensics` entry point must exist and run under the
interpreter verified in section 2b.

In [ ]:
cli_path = shutil.which("aiforensics")
print("aiforensics on PATH:", cli_path or "<none>")

if not cli_path:
    raise RuntimeError(
        "The aiforensics CLI is not on PATH. Re-run the install cell, and make "
        "sure the Python 3.10 environment from section 2 is still first on PATH."
    )

result = subprocess.run([cli_path, "--help"], capture_output=True, text=True, check=False)
if result.returncode != 0:
    raise RuntimeError(f"aiforensics --help failed with exit code {result.returncode}")

print("OK: CLI available at", cli_path)

## 5. Storage inputs

Kaggle separates **read-only** attached data from **writable** working storage:

- `/kaggle/input/<your-dataset>` is read-only. Research images and pre-built
  manifests normally live here.
- `/kaggle/working` is writable but ephemeral; save the notebook output to keep
  anything beyond the session.

No dataset slug or username is hardcoded. Replace the `<...>` placeholders with
the datasets you attached.

In [ ]:
# User-editable: Kaggle mount points.
KAGGLE_INPUT_ROOT = Path("/kaggle/input")  # read-only attached datasets
KAGGLE_WORKING_ROOT = Path("/kaggle/working")  # writable, ephemeral

# Full path to the attached dataset directory that holds the images. Kaggle
# mounts datasets under different shapes (/kaggle/input/<slug> and
# /kaggle/input/datasets/<owner>/<slug> both occur), so give the whole path
# instead of assuming one layout.
INPUT_DATA_DIR = KAGGLE_INPUT_ROOT / "<your-images-dataset>"
INPUT_CHECKPOINT_DIR = KAGGLE_INPUT_ROOT / "<your-npr-checkpoint-dataset>"

# Set True to build manifests from a GenImage-layout INPUT_DATA_DIR in this
# session; set False when manifest CSVs are already provisioned in an attached
# dataset. Building writes CSV files, and /kaggle/input is read-only, so the two
# modes cannot share the same manifest root.
BUILD_MANIFESTS = True
INPUT_MANIFEST_DIR = KAGGLE_INPUT_ROOT / "<your-manifests-dataset>"

# Set True to keep HuggingFace model weights (~6 GB for Qwen) inside
# /kaggle/working, so saved notebook output restores them without a re-download
# after a session reset. Costs output quota (~6 GB of ~20 GB) and makes saving
# slower. False keeps the default ephemeral cache under /root/.cache.
PERSIST_HF_CACHE = True
if PERSIST_HF_CACHE:
    HF_CACHE_ROOT = KAGGLE_WORKING_ROOT / "hf-cache"
    HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    # The %%bash run cells inherit this, so the CLI's transformers downloads
    # land in persistent storage too.
    os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
else:
    HF_CACHE_ROOT = None

# Read-only inputs.
DATA_ROOT = INPUT_DATA_DIR
NPR_CHECKPOINT_PATH = INPUT_CHECKPOINT_DIR / "NPR.pth"

# Writable outputs: never place these under /kaggle/input.
CACHE_ROOT = KAGGLE_WORKING_ROOT / "cache"
OUTPUT_ROOT = KAGGLE_WORKING_ROOT / "outputs"
EXTERNAL_ROOT = KAGGLE_WORKING_ROOT / "external"

# Built manifests must land in writable storage; provisioned ones stay read-only.
MANIFEST_ROOT = KAGGLE_WORKING_ROOT / "manifests" if BUILD_MANIFESTS else INPUT_MANIFEST_DIR

# Manifest filenames are separate config fields today; override them if your
# provisioned manifests use different names.
TINY_TRAIN_MANIFEST = MANIFEST_ROOT / "tiny_genimage_train.csv"
TINY_DEV_MANIFEST = MANIFEST_ROOT / "tiny_genimage_dev.csv"
GENIMAGE_UNSEEN_MANIFEST = MANIFEST_ROOT / "genimage_unseen_external.csv"
SYNTHBUSTER_MANIFEST = MANIFEST_ROOT / "synthbuster_external.csv"

writable_roots = [CACHE_ROOT, OUTPUT_ROOT, EXTERNAL_ROOT]
if BUILD_MANIFESTS:
    writable_roots.append(MANIFEST_ROOT)
for writable in writable_roots:
    writable.mkdir(parents=True, exist_ok=True)

if not BUILD_MANIFESTS and MANIFEST_ROOT.is_relative_to(KAGGLE_INPUT_ROOT):
    print("manifest root is read-only; manifests must already exist there")

print("data root    : (read-only)", DATA_ROOT)
print("manifest root:", "(writable)" if BUILD_MANIFESTS else "(read-only)", MANIFEST_ROOT)
print("npr ckpt     : (read-only)", NPR_CHECKPOINT_PATH)
print("hf cache     :", HF_CACHE_ROOT if HF_CACHE_ROOT else "(default, ephemeral)")
print("cache root   : (writable)", CACHE_ROOT)
print("output root  : (writable)", OUTPUT_ROOT)
print("external root: (writable)", EXTERNAL_ROOT)
print("build manifests:", BUILD_MANIFESTS)

## 6. Generate the runtime config

`configs/phase_ab.yaml` is treated as a **read-only template**. This cell copies
it and rewrites path values only, then writes the result under
`.cache/aiforensics-notebook/` inside the repository.

Two constraints drive that location:

- the config loader finds the repository root by walking up to `pyproject.toml`,
  so the generated file must stay under `REPO_ROOT`;
- `.cache/` is git-ignored, so the generated config is never committed.

Scientific settings are **not** touched: dataset enable flags, model ids, prompt
ids, CLIP seeds, the metric list, report policy, and the pinned NPR repository
URL/commit all stay exactly as committed.

In [ ]:
import yaml

TEMPLATE_CONFIG = REPO_ROOT / "configs/phase_ab.yaml"
GENERATED_CONFIG = REPO_ROOT / ".cache/aiforensics-notebook/phase_ab_kaggle.yaml"

with open(TEMPLATE_CONFIG, encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

# Only environment/path values change.
cfg["paths"]["data_root"] = str(DATA_ROOT)
cfg["paths"]["manifest_root"] = str(MANIFEST_ROOT)
cfg["paths"]["cache_root"] = str(CACHE_ROOT)
cfg["paths"]["output_root"] = str(OUTPUT_ROOT)
cfg["paths"]["external_root"] = str(EXTERNAL_ROOT)

cfg["datasets"]["tiny_genimage"]["train_manifest"] = str(TINY_TRAIN_MANIFEST)
cfg["datasets"]["tiny_genimage"]["dev_manifest"] = str(TINY_DEV_MANIFEST)
cfg["datasets"]["genimage_unseen"]["manifest"] = str(GENIMAGE_UNSEEN_MANIFEST)
cfg["datasets"]["synthbuster"]["manifest"] = str(SYNTHBUSTER_MANIFEST)

cfg["baselines"]["npr"]["checkpoint_path"] = str(NPR_CHECKPOINT_PATH)

GENERATED_CONFIG.parent.mkdir(parents=True, exist_ok=True)
with open(GENERATED_CONFIG, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

os.environ["AIF_CONFIG"] = str(GENERATED_CONFIG)
print("runtime config:", GENERATED_CONFIG)

## 7. Validate provisioned inputs

This cell only reports where things are. It does not read image data, compute
checksums, or validate manifest contents: `aiforensics prepare` remains the
authoritative validator.

When `BUILD_MANIFESTS` is true the manifest CSVs are expected to be **absent**
here; section 8 creates them. The cell also lists the generator directories it
can see under `DATA_ROOT` and checks them against the generators the config asks
for, because a wrong `DATA_ROOT` is the most common first-run mistake.

A missing NPR checkpoint is surfaced here, before the NPR command runs. Do not
download a checkpoint from an unverified third party.

In [ ]:
enabled_manifests = []
if cfg["datasets"]["tiny_genimage"]["enabled"]:
    enabled_manifests += [TINY_TRAIN_MANIFEST, TINY_DEV_MANIFEST]
if cfg["datasets"]["genimage_unseen"]["enabled"]:
    enabled_manifests.append(GENIMAGE_UNSEEN_MANIFEST)
if cfg["datasets"]["synthbuster"]["enabled"]:
    enabled_manifests.append(SYNTHBUSTER_MANIFEST)

print("runtime config :", GENERATED_CONFIG, "exists:", GENERATED_CONFIG.is_file())
print("data root      :", DATA_ROOT, "exists:", DATA_ROOT.is_dir())
print("manifest root  :", MANIFEST_ROOT, "exists:", MANIFEST_ROOT.is_dir())
print("cache root     :", CACHE_ROOT, "exists:", CACHE_ROOT.is_dir())
print("output root    :", OUTPUT_ROOT, "exists:", OUTPUT_ROOT.is_dir())
print("external root  :", EXTERNAL_ROOT, "exists:", EXTERNAL_ROOT.is_dir())

# Generator directories are the dataset layout contract: a directory holding at
# least one of the dataset-native split directories.
split_dirs = ("train", "val")
found_generators = []
if DATA_ROOT.is_dir():
    for entry in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
        if any((entry / split).is_dir() for split in split_dirs):
            found_generators.append(entry.name)

print("\ngenerator directories under data root:", len(found_generators))
for name in found_generators:
    print("  -", name)
if not found_generators and BUILD_MANIFESTS:
    print(
        "  none found: check DATA_ROOT. Expected "
        "<DATA_ROOT>/<generator>/<train|val>/<ai|nature>/"
    )

configured_generators = []
if cfg["datasets"]["tiny_genimage"]["enabled"]:
    configured_generators += cfg["datasets"]["tiny_genimage"].get("generators", [])
if cfg["datasets"]["genimage_unseen"]["enabled"]:
    configured_generators += cfg["datasets"]["genimage_unseen"].get("generators", [])

missing_generators = [g for g in configured_generators if g not in found_generators]
print("\nconfigured generators:", configured_generators or "none")
if missing_generators:
    print("  MISSING under data root:", missing_generators)
    print("  prepare --build-manifests will fail until DATA_ROOT or the config matches")

print("\nmanifests for enabled datasets:")
for path in enabled_manifests:
    print("  -", path, "exists:", path.is_file())
if BUILD_MANIFESTS:
    print("  (BUILD_MANIFESTS is true: section 8 creates/overwrites these)")

print("\nnpr checkpoint :", NPR_CHECKPOINT_PATH, "exists:", NPR_CHECKPOINT_PATH.is_file())
if not NPR_CHECKPOINT_PATH.is_file():
    print(
        "  NPR will follow its allow_deferred policy: provide the official "
        "checkpoint at this path for a real NPR run."
    )

## 8. Full Phase A/B run

These cells call the public CLI in the required order. Each cell uses
`set -euo pipefail`, so a real CLI failure stops the cell and stays visible.
Nothing is wrapped in `|| true`.

The first cell runs `prepare`. When `BUILD_MANIFESTS` is true it passes
`--build-manifests`, which reads the GenImage-layout `DATA_ROOT`
(`<generator>/<train|val>/<ai|nature>/`) and **overwrites** the configured
manifest CSVs before validating them. Which generators are in-distribution
versus held out comes from `configs/phase_ab.yaml`, not from this notebook.

`assisted_qwen` must run **after** `clip_probe` because its Phase A/B contract
consumes CLIP assistant predictions.

One `clip_probe` command covers every configured seed; the notebook never loops
over seeds itself. A baseline that records a `deferred` artifact according to its
adapter contract is a legitimate environment outcome, not a notebook error to
swallow.

In [ ]:
# AIF_SECTION: full_run
os.environ["AIF_PREPARE_ARGS"] = "--build-manifests" if BUILD_MANIFESTS else ""
print("prepare args:", os.environ["AIF_PREPARE_ARGS"] or "(validate only)")

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics prepare ${AIF_PREPARE_ARGS} --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics run --baseline clip_probe --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics run --baseline qwen_vl --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics run --baseline npr --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics run --baseline assisted_qwen --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics evaluate --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics report --config "$AIF_CONFIG"

## 9. Artifacts

Where to look after a run. The notebook only points at these files; parsing and
rendering stay in the package (`aiforensics evaluate` and `aiforensics report`).

```text
<OUTPUT_ROOT>/manifest_validation.json
<OUTPUT_ROOT>/<run_id>/status.json
<OUTPUT_ROOT>/<run_id>/predictions.jsonl
<OUTPUT_ROOT>/<run_id>/metrics.json
<OUTPUT_ROOT>/<run_id>/metrics_by_source.csv
<OUTPUT_ROOT>/<configured report filename>
```

In [ ]:
report_path = OUTPUT_ROOT / cfg["report"]["filename"]

print("manifest validation:", OUTPUT_ROOT / "manifest_validation.json")
print("report             :", report_path, "exists:", report_path.is_file())

print("\nrun directories under", OUTPUT_ROOT)
if OUTPUT_ROOT.is_dir():
    for entry in sorted(p for p in OUTPUT_ROOT.iterdir() if p.is_dir()):
        print("  -", entry.name)

## 10. Optional: smoke verification

The smoke flow proves that installation and the CLI pipeline work in this hosted
environment. It uses the committed `configs/phase_ab_smoke.yaml` fixtures.

> Smoke metrics are pipeline checks, **not scientific evidence**.

The committed smoke config is never modified. This section generates a copy that
relocates `cache_root` and `output_root` only, so nothing is written to
repository paths that may be read-only or ephemeral. Smoke `data_root` and smoke
manifests keep pointing at the repository fixtures, because those fixtures *are*
the smoke dataset.

In [ ]:
# AIF_SECTION: smoke
SMOKE_TEMPLATE = REPO_ROOT / "configs/phase_ab_smoke.yaml"
SMOKE_CONFIG = REPO_ROOT / ".cache/aiforensics-notebook/phase_ab_smoke_kaggle.yaml"

with open(SMOKE_TEMPLATE, encoding="utf-8") as handle:
    smoke_cfg = yaml.safe_load(handle)

# Relocate writable roots only; fixtures stay where they are committed.
smoke_cfg["paths"]["cache_root"] = str(CACHE_ROOT / "smoke")
smoke_cfg["paths"]["output_root"] = str(OUTPUT_ROOT / "smoke")

SMOKE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
with open(SMOKE_CONFIG, "w", encoding="utf-8") as handle:
    yaml.safe_dump(smoke_cfg, handle, sort_keys=False)

os.environ["AIF_SMOKE_CONFIG"] = str(SMOKE_CONFIG)
print("smoke runtime config:", SMOKE_CONFIG)

In [ ]:
%%bash
# AIF_SECTION: smoke
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics prepare --config "$AIF_SMOKE_CONFIG"
aiforensics run --baseline clip_probe --config "$AIF_SMOKE_CONFIG"
aiforensics run --baseline qwen_vl --config "$AIF_SMOKE_CONFIG"
aiforensics run --baseline npr --config "$AIF_SMOKE_CONFIG"
aiforensics run --baseline assisted_qwen --config "$AIF_SMOKE_CONFIG"
aiforensics evaluate --config "$AIF_SMOKE_CONFIG"
aiforensics report --config "$AIF_SMOKE_CONFIG"